# Verifying Quicopt's Maximum Independent Set solutions

This notebook independently checks every published solution **certificate** in
[`../solutions/mis.json`](../solutions/mis.json). For each QOBLIB graph it recomputes the set
directly from the graph and confirms that it is a valid independent set whose size equals both
the certificate's stated value and the published leaderboard
([`../../data/mis-gpu.csv`](../../data/mis-gpu.csv) / [`../../data/mis-cpu.csv`](../../data/mis-cpu.csv)).

**What a certificate is.** One bitstring per instance — the single best independent set Quicopt
found. `bits[i]='1'` means vertex $i$ is in the set $S$, `'0'` means it is not. A *valid* MIS
certificate must satisfy two things:

$$\text{independence:}\quad y_i\,y_j = 0 \ \ \forall (i,j)\in E, \qquad\qquad \text{size:}\quad |S| = \sum_i y_i,$$

where $y_i\in\{0,1\}$ is the membership bit. The first says no edge joins two selected vertices;
the second is the objective we report. The set size is reported on two back-ends (GPU/CPU); the
certificate is the larger of the two, so we cross-check it against the **best** leaderboard value.

**The instances.** We do not redistribute the QOBLIB graphs — they are the standard public
benchmark set ([`07-independentset`](https://git.zib.de/qopt/qoblib-quantum-optimization-benchmarking-library)).
Point `MIS_DIR` at your own copy, or run the optional download cell to fetch them from the
canonical source at the pinned commit the records were run on.

In [ ]:
import json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd()                                   # Jupyter runs from this notebook's dir
SOLUTIONS = HERE / ".." / "solutions" / "mis.json"
GPU_LB    = HERE / ".." / ".." / "data" / "mis-gpu.csv"
CPU_LB    = HERE / ".." / ".." / "data" / "mis-cpu.csv"

# Where the QOBLIB .gph instances live. We do NOT ship them (standard public benchmark set).
# Point this at your local copy, or run the optional download cell below.
MIS_DIR   = HERE / "mis_instances"
QOBLIB_SHA = "18426267491c1f041aba1eebfcca279015668677"   # commit the records were run on
QOBLIB_URL = ("https://git.zib.de/qopt/qoblib-quantum-optimization-benchmarking-library"
              "/-/raw/{sha}/07-independentset/instances/{instance}.gph")

doc  = json.loads(Path(SOLUTIONS).read_text())
sols = doc["solutions"]
print(doc["solver"], "\u2014", len(sols), "certificates")
print(doc["convention"])

### (optional) fetch the QOBLIB instances
Run this only if you don't already have the `.gph` files locally. It downloads any missing
instance into `MIS_DIR` from the canonical public source at the pinned commit. Skip it and set
`MIS_DIR` to your own copy if you prefer.

In [ ]:
MIS_DIR.mkdir(parents=True, exist_ok=True)
for inst in sols:
    f = MIS_DIR / f"{inst}.gph"
    if not f.exists():
        urllib.request.urlretrieve(QOBLIB_URL.format(sha=QOBLIB_SHA, instance=inst), f)
        print("downloaded", inst)
print("instances present:", sum((MIS_DIR / f"{i}.gph").exists() for i in sols), "/", len(sols))

In [ ]:
def parse_gph(path):
    """QOBLIB .gph (DIMACS) -> (N, i, j), 0-indexed. Header 'p edge N M'; rows 'e u v' 1-indexed."""
    N, i, j = None, [], []
    with open(path) as fh:
        for ln in fh:
            t = ln.split()
            if not t:
                continue
            if t[0] == "p":
                N = int(t[2])
            elif t[0] == "e":
                i.append(int(t[1]) - 1)
                j.append(int(t[2]) - 1)
    return N, np.array(i, dtype=np.int64), np.array(j, dtype=np.int64)

def membership(bits):
    """y in {0,1}^N from the certificate bitstring: y_i = 1 iff bits[i] == '1' (vertex i in S)."""
    return (np.frombuffer(bits.encode(), np.uint8) == ord("1")).astype(np.int64)

rows = []
for inst, s in sols.items():
    f = MIS_DIR / f"{inst}.gph"
    if not f.exists():
        rows.append({"instance": inst, "N": s["N"], "claimed": s["size"],
                     "recomputed": None, "independent": None, "match": None})
        continue
    N, i, j = parse_gph(f)
    assert len(s["bits"]) == s["N"] == N, f"{inst}: bits length != N"
    y = membership(s["bits"])
    independent = True if len(i) == 0 else not bool(np.any(y[i] & y[j]))   # no selected-selected edge
    size = int(y.sum())
    rows.append({"instance": inst, "N": s["N"], "claimed": s["size"], "recomputed": size,
                 "independent": independent, "match": independent and size == s["size"]})

In [ ]:
df = pd.DataFrame(rows)

# cross-check each claimed size against the published leaderboard: the certificate is the best of
# the two back-ends, so it must equal max(GPU set size, CPU set size) for that instance.
gpu = pd.read_csv(GPU_LB).set_index("instance")["set size"]
cpu = pd.read_csv(CPU_LB).set_index("instance")["set size"]
df["leaderboard"] = df["instance"].map(lambda n: int(max(gpu.get(n, -1), cpu.get(n, -1))))
df["== leaderboard"] = df["claimed"] == df["leaderboard"]

checked = df[df["recomputed"].notna()]
print(f"verified {len(checked)}/{len(df)} instances"
      + ("" if len(checked) == len(df) else "  (others: instance file not found in MIS_DIR)"))
print("all sets independent:                 ", bool(checked["independent"].all()))
print("all recomputed sizes == certificate:  ", bool((checked["recomputed"] == checked["claimed"]).all()))
print("all certificates  == leaderboard best:", bool(df["== leaderboard"].all()))
assert checked["independent"].all(), "a published set is not independent!"
assert (checked["recomputed"] == checked["claimed"]).all(), "a recomputed size disagrees with its certificate!"
assert df["== leaderboard"].all(), "a certificate disagrees with the published leaderboard!"
df